# runtimed Python API Redesign

Exploring a better API surface for `runtimed`'s notebook rooms and cells. The current API exposes low-level primitives — raw dicts from `list_rooms()`, flat `get_cell_*/set_cell_*` methods on sessions. This notebook designs a higher-level API that respects the automerge local-first architecture underneath.

In [ ]:
import runtimed

In [ ]:
client = runtimed.AsyncClient()
sync_client = runtimed.Client()

# Cells API v2

## Design principles

1. **Reads are sync.** The notebook is a local-first automerge document — `get_cell`, `get_cell_source`, etc. are reading from the local replica, not doing I/O. The API should reflect this: cell properties should be sync reads, not futures.

2. **Writes are async.** Setting source, creating cells, deleting — these mutate the automerge doc and sync to peers. Even though the local write is fast, the API should be async to reflect that it's a distributed operation.

3. **Cells live on the session.** A session *is* your connection to a notebook document. `session.cells` is the natural place to access the cell collection — no separate construction needed.

4. **Cell access by ID, not index.** Cell IDs are stable (they survive reorders, concurrent edits). Positional indexing is fragile in a collaborative doc. Use `.cells[cell_id]` for direct access, iteration for discovery.

5. **CellProxy is a live handle.** It's a reference into the document, not a snapshot. Accessing `.source` reads the current state. This matches how you'd think about a node in a CRDT doc.

## The sync/async duality

The native `AsyncSession` is a PyO3 object — we can't add `.cells` to it from Python. But this reveals something about the ideal design: the session should **internally** hold both the sync doc reader and the async connection. The `CellCollection` and `CellHandle` demonstrate the pattern:

- **Sync reads** via properties (`.source`, `.cell_type`, `.outputs`) — these are just looking at the local automerge replica
- **Async writes** via methods (`.set_source()`, `.append()`, `.run()`) — these send changes that sync to peers
- **Sync collection ops** (`cells[id]`, `for c in cells`, `cells.find(...)`) — all reading from the local doc
- **Async mutations** (`await cells.create(...)`) — these change the document

This is the right split. The native `Session`/`AsyncSession` could expose a `.cells` property that returns a `CellCollection` backed by the same internal automerge doc handle. No need for the user to manually pair sync and async sessions.

## What the ideal API looks like end-to-end

```python
nb = Notebooks(client)
room = await nb.find("gremlins")     # NotebookRoom — inspectable before joining
session = await room.join()          # NotebookSession wrapping the native session

# Cells as a document collection
session.cells                        # CellCollection — sync reads, async writes
session.cells.getById("2be7a0bd-..") # CellHandle by exact ID — sync
session.cells.getByIndex(0)          # CellHandle by position — sync
session.cells.getByIndex(-1)         # last cell
len(session.cells)                   # sync
for cell in session.cells:           # sync iteration
    print(cell.source)               # sync property read

# Searching
session.cells.find("import")        # sync — scans local doc

# Mutations
cell = await session.cells.create("x = 1")  # async — modifies doc
await cell.set_source("x = 2")              # async
await cell.append("\ny = 3")                 # async (automerge splice)
await cell.run()                             # async — triggers execution
await cell.delete()                          # async
```

Note what's missing: no `get_cell_source(id)`, `set_cell_source(id, ...)`, `get_cell_type(id)`, etc. The flat `session.verb_noun(id, ...)` surface collapses into object-oriented access on handles. The session goes from ~20 cell methods to just `session.cells`.

### `getById` vs `getByIndex`

Access is explicit — no overloaded `[]` guessing whether you passed an ID or index. `getById` is the primary access (IDs are stable in a CRDT). `getByIndex` is a convenience for "first cell", "last cell" scripting.

## Execution model insight

You can't `execute_cell` in your own kernel — it deadlocks (the cell being executed can't start until the calling cell finishes). This means:

- **Self-notebook**: `cell.run()` / `cell.queue()` is for controlling *other* notebooks from a driver notebook, or for external scripts/MCP tools
- **Same-notebook**: Programmatic cell interaction is about **editing the document** — creating cells, modifying source, reorganizing — not executing

This is actually clean separation. The API has two distinct use cases:
1. **Document editing** — manipulate cells in any notebook you've joined (sync reads, async writes)
2. **Remote execution** — drive another notebook's kernel (always async, returns results)

For the `session.cells` API, both are available. But a user working *inside* the notebook will mostly use the document-editing side.

## Access patterns: `getById` / `getByIndex`

Instead of overloading `cells["2be7"]` to guess whether it's an ID or prefix, make it explicit:

```python
session.cells.getById("2be7a0bd-607c-...")    # exact ID
session.cells.getByIndex(0)                    # positional
session.cells.getByIndex(-1)                   # last cell
```

Both are sync reads from the local doc. Index access is handy for quick scripting even though IDs are the stable reference in a CRDT.

### `__getitem__` — string keys only

`cells[id]` is sugar for `getById`. No int indexing in `[]` — keeps it unambiguous. Use `getByIndex` for positional.

In [ ]:
from typing import Optional, List
from dataclasses import dataclass, field
from pathlib import Path


@dataclass
class NotebookRoom:
    """A notebook room you can inspect before joining."""

    notebook_id: str
    kernel_type: Optional[str] = None
    kernel_status: Optional[str] = None
    active_peers: int = 0
    has_kernel: bool = False
    env_source: Optional[str] = None
    _client: Optional[object] = field(default=None, repr=False)

    @property
    def path(self) -> Optional[Path]:
        if "/" in self.notebook_id:
            return Path(self.notebook_id)
        return None

    @property
    def name(self) -> str:
        return self.path.stem if self.path else self.notebook_id[:8]

    @property
    def is_ephemeral(self) -> bool:
        return self.path is None

    async def join(self, peer_label=None):
        if self._client is None:
            raise RuntimeError("Room not bound to a client")
        return await self._client.join_notebook(self.notebook_id, peer_label)

    def __repr__(self):
        status = f" [{self.kernel_status}]" if self.kernel_status else ""
        peers = f" ({self.active_peers} peers)" if self.active_peers else ""
        return f"NotebookRoom({self.name!r}{status}{peers})"


def _parse_room(raw: dict, client=None) -> NotebookRoom:
    return NotebookRoom(
        notebook_id=raw["notebook_id"],
        kernel_type=raw.get("kernel_type"),
        kernel_status=raw.get("kernel_status"),
        active_peers=int(raw.get("active_peers", 0)),
        has_kernel=raw.get("has_kernel") == "true",
        env_source=raw.get("env_source"),
        _client=client,
    )


class Notebooks:
    """Discover and connect to notebook rooms."""

    def __init__(self, client):
        self._client = client

    async def list(self) -> List[NotebookRoom]:
        raw = await self._client.list_rooms()
        return [_parse_room(r, self._client) for r in raw]

    async def find(self, query: str) -> Optional[NotebookRoom]:
        for room in await self.list():
            if (
                query.lower() in room.notebook_id.lower()
                or query.lower() in room.name.lower()
            ):
                return room
        return None

    async def active(self) -> List[NotebookRoom]:
        return [r for r in await self.list() if r.active_peers > 0]


class CellHandle:
    """A live reference to a cell in the notebook document.
    Reads are sync (local automerge doc). Writes are async (synced to peers)."""

    def __init__(self, cell_id: str, async_session, sync_session):
        self._id = cell_id
        self._async = async_session
        self._sync = sync_session

    @property
    def id(self) -> str:
        return self._id

    @property
    def source(self) -> str:
        return self._sync.get_cell_source(self._id)

    @property
    def cell_type(self) -> str:
        return self._sync.get_cell_type(self._id)

    @property
    def outputs(self):
        return self._sync.get_cell(self._id).outputs

    @property
    def execution_count(self):
        return self._sync.get_cell_execution_count(self._id)

    @property
    def metadata(self) -> dict:
        return self._sync.get_cell_metadata(self._id)

    def snapshot(self):
        return self._sync.get_cell(self._id)

    async def set_source(self, source: str) -> "CellHandle":
        await self._async.set_source(self._id, source)
        return self

    async def append(self, text: str) -> "CellHandle":
        await self._async.append_source(self._id, text)
        return self

    async def splice(
        self, index: int, delete_count: int, text: str = ""
    ) -> "CellHandle":
        await self._async.splice_source(self._id, index, delete_count, text)
        return self

    async def run(self, timeout_secs: float = 60.0):
        return await self._async.execute_cell(self._id, timeout_secs)

    async def queue(self) -> "CellHandle":
        await self._async.queue_cell(self._id)
        return self

    async def delete(self):
        await self._async.delete_cell(self._id)

    async def move_after(self, other=None) -> "CellHandle":
        after_id = other._id if other else None
        await self._async.move_cell(self._id, after_id)
        return self

    def __repr__(self):
        return f"Cell({self._id[:8]}, {self.cell_type})"


class CellCollection:
    """The cells in a notebook. Sync reads, async mutations."""

    def __init__(self, async_session, sync_session):
        self._async = async_session
        self._sync = sync_session

    def _handle(self, cell_id) -> CellHandle:
        return CellHandle(cell_id, self._async, self._sync)

    def getById(self, cell_id: str) -> CellHandle:
        ids = self._sync.get_cell_ids()
        if cell_id not in ids:
            raise KeyError(f"No cell with ID {cell_id!r}")
        return self._handle(cell_id)

    def getByIndex(self, index: int) -> CellHandle:
        ids = self._sync.get_cell_ids()
        return self._handle(ids[index])

    def __iter__(self):
        for cid in self._sync.get_cell_ids():
            yield self._handle(cid)

    def __len__(self) -> int:
        return len(self._sync.get_cell_ids())

    def __contains__(self, cell_id: str) -> bool:
        return cell_id in self._sync.get_cell_ids()

    @property
    def ids(self) -> list:
        return self._sync.get_cell_ids()

    def find(self, substring: str) -> List[CellHandle]:
        return [
            self._handle(cid)
            for cid in self._sync.get_cell_ids()
            if substring in self._sync.get_cell_source(cid)
        ]

    async def create(
        self, source: str = "", cell_type: str = "code", index=None
    ) -> CellHandle:
        cell_id = await self._async.create_cell(source, cell_type, index)
        return self._handle(cell_id)

    def __repr__(self):
        return f"Cells({len(self)})"


print("✓ API classes loaded")

## Room discovery

Browse active notebooks, inspect them, and join.

In [ ]:
nb = Notebooks(client)

# List all rooms
rooms = await nb.list()
for room in rooms:
    print(f"  {room}  ephemeral={room.is_ephemeral}  path={room.path}")

In [ ]:
# Find and inspect a room by name
room = await nb.find("gremlins")
print(room)
print(f"  name: {room.name}")
print(f"  path: {room.path}")
print(f"  kernel: {room.kernel_type} ({room.kernel_status})")
print(f"  peers: {room.active_peers}")

## Working with cells

Create a separate notebook and use the cells API to build it up programmatically.

In [ ]:
# Create a fresh notebook to play with
async_s = await client.create_notebook()
playground_id = async_s.notebook_id
sync_s = sync_client.join_notebook(playground_id)

cells = CellCollection(async_s, sync_s)
print(f"Created playground notebook: {playground_id[:8]}")
print(f"Cells: {cells}")

In [ ]:
# Build up a notebook programmatically
import time

title = await cells.create(
    "# Playground Notebook\nCreated by the cells API", cell_type="markdown"
)
setup = await cells.create("import math")
calc = await cells.create(
    "# Calculate some values\nresults = [math.sqrt(n) for n in range(1, 6)]\nresults"
)

# Give sync session a moment to receive the automerge updates
time.sleep(0.3)

print(f"Created {len(cells)} cells:")
for cell in cells:
    print(f"  {cell}: {cell.source.splitlines()[0]!r}")

In [ ]:
# Execute cells in the playground's kernel (cross-notebook!)
result = await setup.run()
print(f"Setup: {result}")

result = await calc.run()
print(f"Calc: {result}")

# Read the outputs (sync!)
time.sleep(0.2)
print(f"\nOutputs: {calc.outputs}")

In [ ]:
# Edit a cell and re-run it
await calc.set_source(
    "# Golden ratio\nphi = (1 + math.sqrt(5)) / 2\nprint(f'φ = {phi}')"
)

result = await calc.run()
time.sleep(0.2)

print(f"Updated source: {calc.source!r}")
print(f"Output: {calc.outputs[0].text}")

In [ ]:
# Search and iterate
print(f"Total cells: {len(cells)}")
print(f"Cells with 'math': {cells.find('math')}")
print()

# Access by index
first = cells.getByIndex(0)
last = cells.getByIndex(-1)
print(f"First: {first} → {first.source.splitlines()[0]!r}")
print(f"Last:  {last} → {last.source.splitlines()[0]!r}")

## Runtime state

The RuntimeStateDoc is a readonly automerge doc that tells you what's happening: kernel status, execution queue, environment sync state. Let's see what's available.

In [ ]:
# Explore the runtime state on our playground session
rs = await async_s.get_runtime_state()
print(f"RuntimeState: {rs}")
print(f"  type: {type(rs)}")
print(f"  attrs: {[a for a in dir(rs) if not a.startswith('_')]}")
print()

qs = await async_s.get_queue_state()
print(f"QueueState: {qs}")
print(f"  executing: {qs.executing}")
print(f"  queued: {qs.queued}")

In [ ]:
# Dig into the sub-states
print("--- rs.kernel ---")
k = rs.kernel
print(f"  type: {type(k)}")
print(f"  attrs: {[a for a in dir(k) if not a.startswith('_')]}")
print(f"  value: {k}")

print("\n--- rs.queue ---")
q = rs.queue
print(f"  type: {type(q)}")
print(f"  attrs: {[a for a in dir(q) if not a.startswith('_')]}")
print(f"  value: {q}")

print("\n--- rs.env ---")
e = rs.env
print(f"  type: {type(e)}")
print(f"  attrs: {[a for a in dir(e) if not a.startswith('_')]}")
print(f"  value: {e}")

print("\n--- rs.last_saved ---")
print(f"  value: {rs.last_saved}")

In [ ]:
# Let's see the kernel state and env state in more detail
print("KernelState:")
print(f"  status: {k.status}")
print(f"  language: {k.language}")
print(f"  name: {k.name}")
print(f"  env_source: {k.env_source}")

print("\nEnvState:")
print(f"  in_sync: {e.in_sync}")
print(f"  added: {e.added}")
print(f"  removed: {e.removed}")
print(f"  channels_changed: {e.channels_changed}")
print(f"  deno_changed: {e.deno_changed}")

In [ ]:
# Let's see runtime state while something is executing
# Queue a cell, then immediately check state
await calc.queue()
rs_busy = await async_s.get_runtime_state()
print(f"During execution: {rs_busy}")
print(f"  kernel: {rs_busy.kernel.status}")
print(f"  queue.executing: {rs_busy.queue.executing}")
print(f"  queue.queued: {rs_busy.queue.queued}")

In [ ]:
# Peers — who's connected to the playground?
peers = await async_s.get_peers()
print(f"Peers: {peers}")
print(f"  type: {type(peers)}")
if peers:
    print(f"  first peer type: {type(peers[0])}")
    print(f"  first peer attrs: {[a for a in dir(peers[0]) if not a.startswith('_')]}")

In [ ]:
# Let's also check open_notebook and the create flow
print("--- client methods ---")
import inspect

for m in ["create_notebook", "open_notebook", "join_notebook"]:
    fn = getattr(client, m)
    try:
        sig = inspect.signature(fn)
    except (ValueError, TypeError):
        sig = "(?)"
    print(f"  client.{m}{sig}")

## Runtime state

The `RuntimeStateDoc` is a separate, readonly automerge doc. It has three sub-states:

| State | Type | What it tells you |
|-------|------|-------------------|
| `kernel` | `KernelState` | `status` (idle/busy), `language`, `name`, `env_source` |
| `queue` | `QueueState` | `executing` (cell ID or None), `queued` (list of cell IDs) |
| `env` | `EnvState` | `in_sync`, `added`, `removed`, `channels_changed` |

Plus `last_saved` (timestamp or None).

### Current API
```python
rs = await session.get_runtime_state()
rs.kernel.status    # 'idle'
rs.queue.executing  # 'cell-40ddb4a5-...' or None
rs.env.in_sync      # True
```

This is already reasonably clean — it's structured, typed, and the sub-states are real objects. The main improvement would be making it a sync read (it's reading a local automerge doc) and surfacing it as a property:

### Proposed
```python
session.runtime.kernel.status     # sync read
session.runtime.queue.executing   # sync read  
session.runtime.env.in_sync       # sync read
session.runtime.last_saved        # sync read
```

No `get_` prefix, no await. It's your local view of what the runtime is doing.

## Notebook lifecycle: `Notebooks` as the entry point

The client currently has three ways to get a session:

| Method | Input | Creates new? |
|--------|-------|-------------|
| `create_notebook(runtime, working_dir)` | nothing | Yes — ephemeral notebook |
| `open_notebook(path)` | file path | Yes if not already open |
| `join_notebook(notebook_id)` | ID (UUID or path) | No — must already exist |

### Proposed: `Notebooks` wraps all three

```python
nb = Notebooks(client)

# Discovery (what's already running)
rooms = await nb.list()                    # → List[NotebookRoom]
room = await nb.find("gremlins")           # → NotebookRoom by name/id
active = await nb.active()                 # → rooms with peers

# Opening (from disk)
session = await nb.open("/path/to/nb.ipynb")   # → NotebookSession

# Creating (ephemeral)
session = await nb.create()                     # → NotebookSession
session = await nb.create(runtime="python",     # with options
                          working_dir="/tmp")

# Joining (already running room)
session = await room.join()                     # from a NotebookRoom
session = await nb.join(notebook_id)            # by ID directly
```

### Why wrap `open` and `create`?

They return sessions — same as `join`. But conceptually they're different:
- **open**: "I have a file, give me a session for it"
- **create**: "I want a blank notebook"  
- **join**: "I see a running room, let me in"

All three live naturally on `Notebooks` as the single entry point for getting into a notebook. The client becomes an implementation detail.

## Peers

Currently `get_peers()` returns a list of tuples like `('daemon', 'peer')`. This could be richer:

```python
session.peers                        # sync read → List[Peer]

peer = session.peers[0]
peer.label                           # 'daemon', 'claude', 'kyle-nteract'
peer.role                            # 'peer' | 'daemon' | ...
```

The `peer_label` parameter on `join`/`open`/`create` becomes how you identify yourself to others:

```python
session = await nb.join(room_id, peer_label="api-explorer")
# Other peers can now see "api-explorer" in the peer list
```

This matters for multi-agent / multi-user scenarios — knowing *who* is editing what.

## Full proposed API surface

```
Notebooks(client)
├── .list()                          → List[NotebookRoom]     async
├── .find(query)                     → NotebookRoom | None    async
├── .active()                        → List[NotebookRoom]     async
├── .open(path)                      → NotebookSession        async
├── .create(runtime, working_dir)    → NotebookSession        async
└── .join(notebook_id)               → NotebookSession        async

NotebookRoom
├── .notebook_id                     str
├── .name                            str (stem or short id)
├── .path                            Path | None
├── .is_ephemeral                    bool
├── .kernel_type                     str | None
├── .kernel_status                   str | None
├── .active_peers                    int
├── .has_kernel                      bool
└── .join(peer_label)                → NotebookSession        async

NotebookSession
├── .notebook_id                     str
├── .cells                           CellCollection           sync
│   ├── .getById(cell_id)            → CellHandle             sync
│   ├── .getByIndex(index)           → CellHandle             sync
│   ├── .find(substring)             → List[CellHandle]       sync
│   ├── .ids                         List[str]                sync
│   ├── __iter__                     → Iterator[CellHandle]   sync
│   ├── __len__                      int                      sync
│   ├── __contains__                 bool                     sync
│   └── .create(source, type, index) → CellHandle             async
├── .runtime                         RuntimeState             sync
│   ├── .kernel.status               str
│   ├── .kernel.language             str
│   ├── .queue.executing             str | None
│   ├── .queue.queued                List[str]
│   ├── .env.in_sync                 bool
│   └── .last_saved                  datetime | None
├── .peers                           List[Peer]               sync
├── .save()                                                   async
├── .interrupt()                                              async
├── .restart_kernel()                                         async
└── .close()                                                  async

CellHandle
├── .id                              str                      sync
├── .source                          str                      sync
├── .cell_type                       str                      sync
├── .outputs                         List[Output]             sync
├── .execution_count                 int | None               sync
├── .metadata                        dict                     sync
├── .snapshot()                      → Cell                   sync
├── .set_source(source)              → CellHandle             async
├── .append(text)                    → CellHandle             async
├── .splice(idx, del, text)          → CellHandle             async
├── .set_type(cell_type)             → CellHandle             async
├── .run(timeout)                    → ExecutionResult         async
├── .queue()                         → CellHandle             async
├── .delete()                                                 async
└── .move_after(other)               → CellHandle             async
```